# RMDS Hackathon - Dataset C: COVID-19 Vaccine Prediction & Cost Optimization

This notebook covers our team's end-to-end data science pipeline:
1. **Exploratory Data Analysis (EDA)** & missing value inspection.
2. **Pre-processing Pipeline** with imputation, scaling, and one-hot encoding.
3. **Model Training & Cross-Validation** (Logistic Regression, Random Forest, Gradient Boosting).
4. **Financial Cost Evaluation** (minimizing town expenses for vaccines vs. hospitalizations).
5. **Generation of the Top 5 Submission Files** required for Canvas.

In [16]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Load training and testing datasets
train_df = pd.read_csv('dataset_C_training (2).csv')
test_df = pd.read_csv('dataset_C_testing (1).csv')

print("Training Data Shape:", train_df.shape)
print("Testing Data Shape:", test_df.shape)

Training Data Shape: (4756, 31)
Testing Data Shape: (4749, 30)


## Step 1: Exploratory Data Analysis (EDA)
We inspect the target distribution (`covid_vaccine`) and identify features with missing values.

In [17]:
# Check target distribution
print("Target Variable Distribution (covid_vaccine):")
print(train_df['covid_vaccine'].value_counts(normalize=True))

# Check missing values for top columns
missing_vals = train_df.isnull().sum()
print("\nTop Missing Value Columns:")
print(missing_vals[missing_vals > 0].sort_values(ascending=False).head(10))

Target Variable Distribution (covid_vaccine):
covid_vaccine
0    0.673255
1    0.326745
Name: proportion, dtype: float64

Top Missing Value Columns:
employment_sector               2330
health_insurance                1908
income_poverty                   748
rent_or_own                      329
employment_status                238
marital_status                   231
education                        228
child_under_6_months             135
health_worker                    132
opinion_covid_sick_from_vacc      90
dtype: int64


## Step 2: Data Pre-processing Pipeline
We separate features from the target variable and build a preprocessing pipeline using `ColumnTransformer` to handle numeric and categorical columns appropriately.

In [18]:
# Separate features and target
X = train_df.drop(columns=['respondent_id', 'covid_vaccine'])
y = train_df['covid_vaccine']
X_test = test_df.drop(columns=['respondent_id'])

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing for numeric data: median imputation + scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data: most frequent imputation + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a single preprocessor column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

## Step 3: Model Training & Cross-Validation
We evaluate three core models (Logistic Regression, Random Forest, and Gradient Boosting) using Stratified 5-Fold Cross-Validation.

In [19]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Evaluate using Stratified K-Fold Cross-Validation (ROC-AUC metric)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
    cv_results[name] = {
        'mean_roc_auc': scores.mean(),
        'std_roc_auc': scores.std()
    }

print("Model Cross-Validation ROC-AUC Scores:")
for name, res in cv_results.items():
    print(f"* {name}: Mean ROC-AUC = {res['mean_roc_auc']:.4f} (+/- {res['std_roc_auc']:.4f})")

Model Cross-Validation ROC-AUC Scores:
* Logistic Regression: Mean ROC-AUC = 0.8307 (+/- 0.0081)
* Random Forest: Mean ROC-AUC = 0.8215 (+/- 0.0107)
* Gradient Boosting: Mean ROC-AUC = 0.8390 (+/- 0.0058)


## Step 4: Evaluation Metric & Financial Cost Optimization
To align with the challenge objective, we define a financial cost function ($1,500 per vaccine vs. $41,000 per hospitalization, factoring in an 85% vaccine efficacy reduction). We test different probability decision thresholds to minimize total cost.

In [20]:
def estimate_hospitalization_cost(features_df, predicted_vaccine_status):
    # Baseline hospitalization risk based on chronic conditions and age group
    base_risk = 0.05 + 0.03 * features_df['number_chronic_med_conditions'].fillna(0)
    age_risk_multiplier = features_df['age_group'].apply(lambda x:
        1.5 if x == '65+ Years' else (1.3 if x == '55 - 64 Years' else 1.0))

    unvaccinated_hosp_prob = np.clip(base_risk * age_risk_multiplier, 0.01, 0.40)

    # Vaccine reduces hospitalization risk by 85%
    hosp_prob = np.where(predicted_vaccine_status == 1, unvaccinated_hosp_prob * 0.15, unvaccinated_hosp_prob)

    total_hosp_cost = hosp_prob.sum() * 41000
    total_vaccine_cost = predicted_vaccine_status.sum() * 1500

    return total_hosp_cost + total_vaccine_cost

# Train train/validation split to test threshold tuning
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
gb_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', GradientBoostingClassifier(random_state=42))])
gb_pipe.fit(X_train, y_train)

val_probs = gb_pipe.predict_proba(X_val)[:, 1]
print("Cost Evaluation on Validation Set (Gradient Boosting):")
for thresh in [0.3, 0.4, 0.5]:
    preds = (val_probs >= thresh).astype(int)
    cost = estimate_hospitalization_cost(X_val, preds)
    print(f"Threshold {thresh}: Total Cost = ${cost:,.2f} | Vaccines Distributed = {preds.sum()}")

Cost Evaluation on Validation Set (Gradient Boosting):
Threshold 0.3: Total Cost = $2,399,403.25 | Vaccines Distributed = 384
Threshold 0.4: Total Cost = $2,496,206.55 | Vaccines Distributed = 305
Threshold 0.5: Total Cost = $2,568,192.25 | Vaccines Distributed = 247


## Step 5: Generate the 5 Required Submission Files
We train all models on the full training set and export the 5 required CSV submission files formatted as `challenge_submission_group_X_order_Y.csv`.

In [22]:
# Fit all models on the entire training dataset
trained_pipelines = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X, y)
    trained_pipelines[name] = pipe

# Replace '0' with your actual group number if needed
group_number = 0

# Order 1: Random Forest (standard threshold)
sub1 = pd.DataFrame({'respondent_id': test_df['respondent_id'], 'covid_vaccine': trained_pipelines['Random Forest'].predict(X_test)})
sub1.to_csv(f'challenge_submission_group_{group_number}_order_1.csv', index=False)

# Order 2: Gradient Boosting (standard threshold)
sub2 = pd.DataFrame({'respondent_id': test_df['respondent_id'], 'covid_vaccine': trained_pipelines['Gradient Boosting'].predict(X_test)})
sub2.to_csv(f'challenge_submission_group_{group_number}_order_2.csv', index=False)

# Order 3: Logistic Regression (standard threshold)
sub3 = pd.DataFrame({'respondent_id': test_df['respondent_id'], 'covid_vaccine': trained_pipelines['Logistic Regression'].predict(X_test)})
sub3.to_csv(f'challenge_submission_group_{group_number}_order_3.csv', index=False)

# Order 4: Gradient Boosting with tuned threshold (0.35)
probs_gb = trained_pipelines['Gradient Boosting'].predict_proba(X_test)[:, 1]
sub4 = pd.DataFrame({'respondent_id': test_df['respondent_id'], 'covid_vaccine': (probs_gb >= 0.35).astype(int)})
sub4.to_csv(f'challenge_submission_group_{group_number}_order_4.csv', index=False)

# Order 5: Ensemble average probability threshold at 0.45
ensemble_probs = (
    trained_pipelines['Random Forest'].predict_proba(X_test)[:, 1] +
    trained_pipelines['Gradient Boosting'].predict_proba(X_test)[:, 1] +
    trained_pipelines['Logistic Regression'].predict_proba(X_test)[:, 1]
) / 3.0
sub5 = pd.DataFrame({'respondent_id': test_df['respondent_id'], 'covid_vaccine': (ensemble_probs >= 0.45).astype(int)})
sub5.to_csv(f'challenge_submission_group_{group_number}_order_5.csv', index=False)

print("Successfully generated all 5 submission files in your environment!")

Successfully generated all 5 submission files in your environment!
